# triangle-barycentric — ex2: Cartesian to (u, v) via 2x2 solve

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `triangle-barycentric`. Running the final beacon cell reports progress against the `Geometry: Barycentric coords` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Geometry: Barycentric coords` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`triangle-barycentric`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "triangle-barycentric"
DD_SUBTOPIC = "Geometry: Barycentric coords"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Barycentric coords — inverting Cartesian → (u, v)

Ex1 used the predicate `u >= 0 & v >= 0 & u + v <= 1` on coordinates that were ALREADY in barycentric form. Real pipelines arrive at barycentrics by SOLVING for them: given a point `P` and triangle `ABC`, find the `(u, v)` such that

```
P - A = u * (B - A) + v * (C - A)
```

This is a 2×2 linear system in `(u, v)`:

```
[ (B-A).x  (C-A).x ] [u]   [(P-A).x]
[ (B-A).y  (C-A).y ] [v] = [(P-A).y]
```

**Why a 2×2 not a 3×3 solve.** In 2-D the system has exactly 2 unknowns and 2 equations. In 3-D (Möller-Trumbore) the system becomes 3×3 because the ray parameter `s` joins `(u, v)` as a third unknown.

**Composition with ex1's predicate.** Once you have `(u, v)`, the inside test is *exactly* ex1's three inequalities. Cartesian inside-triangle = solve + predicate.

### Exercise 2 — Cartesian to (u, v) via 2x2 solve

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply a 2x2 `t.linalg.solve` against the edge-basis matrix `[B-A | C-A]` to convert a Cartesian point `P` into barycentric coordinates `(u, v)` w.r.t. triangle `ABC`.
> Keywords: barycentric, linalg-solve, 2x2, inversion
> ```

**KCs targeted:** `barycentric-cartesian-solve`, `barycentric-edge-basis`

Implement `ex2_uv_from_cartesian(A, B, C, P)`.

Inputs (all `(2,)` float tensors): triangle vertices `A`, `B`, `C` and a query point `P`. Return `(u, v)` as a tensor of shape `(2,)` such that `P = A + u*(B-A) + v*(C-A)`.

**Algorithm.** Stack the edge vectors as columns of a 2x2 matrix `M = [B-A | C-A]` and solve `M @ [u, v] = P - A`. One call to `t.linalg.solve(M, P - A)`.

**Hint.** `t.stack([B - A, C - A], dim=1)` gives the 2x2 with edges as columns (NOT rows — that would be the wrong system).

In [ ]:
def ex2_uv_from_cartesian(A: Tensor, B: Tensor, C: Tensor, P: Tensor) -> Tensor:
    M = t.stack([B - A, C - A], dim=1)   # (2, 2), edges as columns
    return t.linalg.solve(M, P - A)       # (2,)


<details><summary>Solution</summary>

```python
def ex2_uv_from_cartesian(A: Tensor, B: Tensor, C: Tensor, P: Tensor) -> Tensor:
    M = t.stack([B - A, C - A], dim=1)   # (2, 2), edges as columns
    return t.linalg.solve(M, P - A)       # (2,)
```

**Why columns, not rows.** Writing the system `u*(B-A) + v*(C-A) = P-A` in matrix form makes `(B-A)` and `(C-A)` the COLUMNS of `M` so that `M @ [u, v]^T` produces the RHS. `t.stack([..., ...], dim=1)` builds the columns; `dim=0` would build rows and silently solve the wrong system.

**Composition with ex1.** Once you have `(u, v)`, the ex1 predicate (`u >= 0 & v >= 0 & u + v <= 1`) is the inside test. Together: cartesian-inside-triangle = this solve + that predicate.

**Failure mode.** A degenerate triangle (three collinear vertices) makes `M` singular and the solve raises `_LinAlgError`. Wrap in `try/except` — exactly the facet drilled in `try-except-solve`.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()